# Ejercicios: universalidad, quines y autorreferencia

La autorreferencia es el núcleo del teorema del punto fijo (Kleene) y de la indecidibilidad
del problema de la parada. En este cuaderno explorarás programas que operan sobre su propio
código, verificarás el teorema de Rice experimentalmente y construirás quines.

**Artículo asociado:** [Universalidad y autorreferencia](../../03-computabilidad/07-universalidad-y-autorreferencia.md)

In [ ]:
import inspect

print('Entorno listo.')

## Ejemplo 1: el quine

Un **quine** es un programa que, sin leer ninguna entrada, produce como salida exactamente
su propio código fuente. Es la instancia más simple del teorema del punto fijo:
existe un programa cuya salida es su propio código.

La técnica estándar usa la **doble representación**: el programa tiene un dato
(su propio código como cadena) y un procedimiento que reconstruye el programa completo.

In [ ]:
# Quine en Python: el programa s = %r\nprint(s %% s)\n imprime su propio código
quine_code   = 's = %r\nprint(s %% s)\n'
quine_output = quine_code % quine_code

print('Código del quine:')
print('---')
print(quine_code.rstrip())
print('---')
print()
print('Salida del quine (debe ser idéntica al código):')
print('---')
print(quine_output.rstrip())
print('---')
print()
print('¿El código coincide con su salida?', quine_code == quine_output)

### Análisis del quine

El quine tiene dos partes:
1. **Datos:** la cadena `s` con `%r` como marcador de posición para sí misma.
2. **Procedimiento:** `print(s % s)` sustituye `%r` por `repr(s)`, generando el código completo.

Cuando se ejecuta `s % s`, Python sustituye `%r` por la representación literal de `s`,
produciendo exactamente la cadena `s = <repr de s>\nprint(s %% s)\n`, que es el código original.

In [ ]:
# Variante: función que devuelve su propio código fuente
def mi_propio_codigo():
    """Devuelve el código fuente de esta propia función."""
    return inspect.getsource(mi_propio_codigo)

codigo_extraido = inspect.getsource(mi_propio_codigo)
codigo_devuelto = mi_propio_codigo()

print('¿La función devuelve su propio código?', codigo_extraido == codigo_devuelto)
print()
print('Código de la función:')
print(codigo_extraido)

## Ejercicio 1: el teorema de Rice en la práctica

El **teorema de Rice** establece que cualquier propiedad no trivial del comportamiento
de un programa es indecidible. Aunque no podemos demostrar esto formalmente en Python,
podemos observar que cualquier analizador estático finito puede ser engañado.

**Propiedad a analizar:** "¿esta función devuelve 42 para alguna entrada?"

Esta es una propiedad del comportamiento (no de la sintaxis), así que por el teorema
de Rice no puede ser decidida en general.

**Tarea:** escribe una función `f_engana(x)` que devuelva 42 solo para un valor
de `x` fuera del rango `[-100, 1000)`, engañando al analizador.

In [ ]:
def analizador_ingenuo_42(f, dominio):
    """Intenta detectar si f(x) == 42 para algún x en dominio."""
    for x in dominio:
        try:
            if f(x) == 42:
                return True, x
        except Exception:
            pass
    return False, None

dominio_prueba = range(-100, 1000)

# TODO: completa esta función para que engañe al analizador
def f_engana(x):
    """Devuelve 42 para algún valor fuera de [-100, 1000), 0 en otro caso."""
    # TODO: completar
    pass

# Descomenta cuando hayas completado f_engana:
# encontrado, testigo = analizador_ingenuo_42(f_engana, dominio_prueba)
# print(f'Analizador sobre [-100,1000): detectado={encontrado}')
# print(f'Prueba directa: f_engana(???) = {f_engana(???)}  # sustituye ??? por tu valor')

### Solución de referencia para el ejercicio 1

In [ ]:
def f1(x): return x * 2        # devuelve 42 si x = 21 (dentro del rango)
def f2(x): return 0            # nunca devuelve 42
def f3(x): return 42           # siempre devuelve 42

def f_engana_ref(x):
    """Solo devuelve 42 para x = 10**18 — el analizador no lo detectará."""
    if x == 10**18:
        return 42
    return 0

dominio_prueba = range(-100, 1000)
funciones = [
    ('f1: 2*x',                    f1),
    ('f2: siempre 0',              f2),
    ('f3: siempre 42',             f3),
    ('f_engana: solo en 10^18',    f_engana_ref),
]

print('Análisis con dominio de prueba [-100, 1000):')
for nombre, f in funciones:
    encontrado, testigo = analizador_ingenuo_42(f, dominio_prueba)
    if encontrado:
        print(f'  {nombre:35}  SÍ detectado (testigo x={testigo})')
    else:
        print(f'  {nombre:35}  NO detectado')

print()
print(f'Verificación: f_engana(10**18) = {f_engana_ref(10**18)}')
print()
print('Conclusión del teorema de Rice:')
print('Ningún analizador con dominio finito puede decidir la propiedad para todos los programas.')
print('Siempre existirá un programa que devuelva 42 solo para una entrada fuera del dominio.')

## Ejercicio 2: la diagonalización de Cantor

La prueba de indecidibilidad del problema de la parada usa la misma estructura que
la demostración de Cantor de que los reales no son numerables.

Aquí simulamos el argumento diagonal para funciones binarias $f_i: \{0,\ldots,n-1\} \to \{0,1\}$.

In [ ]:
funciones_tabla = [
    [1, 0, 1, 0, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [0, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 0, 1, 1, 0, 0, 1],
    [0, 0, 1, 1, 0, 0, 1, 1],
    [1, 1, 0, 0, 1, 1, 0, 0],
    [0, 1, 1, 0, 0, 1, 1, 0],
]
n = len(funciones_tabla)

# La función diagonal: d(i) = 1 - fi(i) niega la diagonal
diagonal = [1 - funciones_tabla[i][i] for i in range(n)]

print('Tabla de funciones con diagonal marcada:')
print(f'     {"  ".join(str(j) for j in range(n))}')
for i, fila in enumerate(funciones_tabla):
    fila_str = '  '.join(f'[{v}]' if j == i else f' {v}' for j, v in enumerate(fila))
    print(f'f{i}: {fila_str}   (diagonal: f{i}({i})={fila[i]})')

print()
print(f'Función diagonal d: {diagonal}')
print()
print('d difiere de cada fi en la posición i → d no está en la lista.')

# Verificar que d es distinta de todas las fi
es_distinta = all(diagonal != fila for fila in funciones_tabla)
print(f'¿d es distinta de todas las f_i de la tabla? {es_distinta}')
print()
print('Conexión con la parada: si H decidiera la parada, construiríamos D que se contradice,')
print('igual que d contradice la pretendida enumeración completa de funciones binarias.')

## Ejemplo 2: puntos fijos numéricos

El teorema del punto fijo de Kleene garantiza que toda transformación computable tiene
un programa que es su propio punto fijo. Aquí exploramos el análogo numérico.

In [ ]:
import math

def punto_fijo_iterativo(f, x0=0.5, iters=1000, tol=1e-10):
    """Busca x tal que f(x) = x por iteración x_{n+1} = f(x_n)."""
    x = x0
    for _ in range(iters):
        x_new = f(x)
        if abs(x_new - x) < tol:
            return x_new, True
        x = x_new
    return x, False

print('Puntos fijos de funciones numéricas (f(x*) = x*):')
print()
funciones_pf = [
    ('cos(x)',            lambda x: math.cos(x),      0.739085),
    ('(x + 2/x)/2',      lambda x: (x + 2/x)/2,      math.sqrt(2)),
    ('1/(1+x)',           lambda x: 1/(1+x),           (math.sqrt(5)-1)/2),
]
for nombre, f, esperado in funciones_pf:
    pf, conv = punto_fijo_iterativo(f)
    if conv:
        ok = '✓' if abs(pf - esperado) < 1e-6 else '✗'
        print(f'  {ok}  f(x) = {nombre:18}  x* = {pf:.8f}  (esperado ≈ {esperado:.8f})')
    else:
        print(f'  f(x) = {nombre:18}  no converge')

print()
print('El teorema del punto fijo de Kleene garantiza lo mismo para programas:')
print('para cualquier transformación computable f, existe P tal que f(<P>) y P se comportan igual.')

## Reflexión

- Los **quines** son la materialización del teorema del punto fijo: programas que reproducen
  su propio código sin leer ninguna entrada.
- El **teorema de Rice** implica que cualquier analizador estático de comportamiento siempre
  puede ser engañado por algún programa — los límites del análisis de programas son fundamentales.
- La **diagonalización** es el argumento unificador: Cantor, Gödel y Turing usan la misma
  estructura lógica para demostrar que ciertas enumeraciones no pueden ser completas.

**Pregunta:** ¿es el siguiente programa un quine válido?
```python
print(open(__file__).read())
```
¿Qué diferencia hay con el quine puro de la celda 1? ¿Qué ocurriría si el archivo
fuente no existe o tiene otro nombre?